# SatMesh — D-LinkNet two-stage training (Kaggle T4)

**Stage 1** — pretrain on DeepGlobe (3-ch RGB) to learn general road structure.
**Stage 2** — fine-tune on Sentinel-2 India (4-ch RGB+NIR) for domain adaptation.

Output: `dlinknet_india.pth` in `/kaggle/working/checkpoints/`.

### Before running
1. Settings → Accelerator → **GPU T4 x2** (or single T4).
2. Settings → Internet → **On** (needed to clone the repo / pip install).
3. Add data: **+ Add Input → Datasets → `balraj98/deepglobe-road-extraction-dataset`**.
4. (Stage 2) Attach a Sentinel-2 India dataset with `*_sat.jpg` + `*_mask.png` (+ optional `*_nir.tif`) tiles, or run the prep cell.

In [ ]:
# 1. Dependencies (Kaggle ships torch/cv2; pin the rest)
!pip -q install "segmentation-models-pytorch>=0.4" "albumentations>=2.0" pyyaml
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.device_count(), 'GPU(s)')

In [ ]:
# 2. Get the SatMesh source
import os, sys
REPO = '/kaggle/working/SatMesh'
if not os.path.exists(REPO):
    !git clone --depth 1 https://github.com/SahilKumar75/SatMesh.git {REPO}
sys.path.insert(0, REPO)
os.chdir(REPO)
from src.model import train as T
print('SatMesh src loaded from', REPO)

In [ ]:
# 3. Locate the DeepGlobe train split (folder containing *_sat.jpg + *_mask.png)
import glob
cands = glob.glob('/kaggle/input/**/train', recursive=True)
DEEPGLOBE = next((c for c in cands if glob.glob(c + '/*_sat.jpg')), None)
assert DEEPGLOBE, 'DeepGlobe train dir not found — add the dataset (Add Input).'
print('DeepGlobe:', DEEPGLOBE, '|', len(glob.glob(DEEPGLOBE + '/*_sat.jpg')), 'tiles')

In [ ]:
# 4. Stage 1 — DeepGlobe pretrain (3-ch). ~25 min on T4 for 30 epochs / 5000 tiles.
cfg = {
  'stage1': {'data_dir': DEEPGLOBE, 'epochs': 30, 'batch': 8, 'lr': 1e-3,
             'img_size': 512, 'subset': 5000, 'use_nir': False,
             'checkpoint_out': '/kaggle/working/checkpoints/stage1.pth'},
  'stage2': {'data_dir': '/kaggle/working/data/sentinel2_india/train', 'epochs': 20,
             'batch': 4, 'lr': 2e-4, 'img_size': 512, 'subset': None, 'use_nir': True,
             'checkpoint_in': '/kaggle/working/checkpoints/stage1.pth',
             'checkpoint_out': '/kaggle/working/checkpoints/dlinknet_india.pth'},
}
T.run_stage(cfg, 'stage1')

In [ ]:
# 5. Stage 2 data — point at an attached Sentinel-2 India dataset, OR generate a
#    small set by rasterizing OSM roads over Sentinel-2 tiles for cities.json bboxes.
#    (Generation needs Internet ON and is slower; the attached-dataset path is preferred.)
import glob, os
attached = glob.glob('/kaggle/input/**/*_sat.jpg', recursive=True)
india_dir = None
if attached:
    # use the parent dir of the first non-DeepGlobe *_sat.jpg
    for f in attached:
        if 'deepglobe' not in f.lower():
            india_dir = os.path.dirname(f); break
if india_dir:
    cfg['stage2']['data_dir'] = india_dir
    print('Stage 2 data:', india_dir)
else:
    print('No attached India dataset. Attempting generation from cities.json...')
    try:
        from src.data.city_config import load_all
        from src.data.sentinel_dl import download_city
        from src.data.mask_raster import build_dataset
        out = '/kaggle/working/data/sentinel2_india/train'; os.makedirs(out, exist_ok=True)
        for cid, c in load_all().items():
            meta = download_city(cid, c.bbox, out)
            build_dataset(cid, c.bbox, (512, 512), out, sat_path=meta.get('rgb'))
        cfg['stage2']['data_dir'] = out
        print('Generated India tiles in', out)
    except Exception as e:
        print('Generation failed (', e, ') — Stage 2 skipped; stage1.pth is the deliverable.')

In [ ]:
# 6. Stage 2 — Sentinel-2 India fine-tune (4-ch). ~20 min on T4.
import glob, shutil, os
have_india = glob.glob(cfg['stage2']['data_dir'] + '/*_sat.jpg')
if have_india:
    T.run_stage(cfg, 'stage2')
else:
    os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
    shutil.copy(cfg['stage1']['checkpoint_out'], '/kaggle/working/checkpoints/dlinknet_india.pth')
    print('No India data — copied stage1 (3-ch) as the checkpoint. Re-run Stage 2 once data is attached.')

In [ ]:
# 7. Deliverable — download from the Output tab
!ls -lh /kaggle/working/checkpoints/
print('Download dlinknet_india.pth and commit it to checkpoints/ in the repo.')